# 🚨 AOG 자재 수급 어시스턴트

좌측 AI 상황 요약 + 우측 6단계 프로세스 진행을 한 화면에서 보여주는 Streamlit 앱입니다. 아래 3개 셀을 순서대로 실행하세요. 사용법·테스트 시나리오는 저장소의 `README.md`를 참고하세요.

## Cell 1 — 라이브러리 설치 (설치 후 런타임 자동 재시작됨, 정상 동작)

Colab은 numpy/pandas가 이미 메모리에 로드된 상태로 시작합니다. 이 상태에서 새 패키지를 설치하면 버전이 어긋나며 위젯 전반에서 TypeError가 나는 것이 Colab의 잘 알려진 증상이라, 설치 직후 런타임을 자동으로 재시작합니다. **"세션이 다운되었습니다" 경고가 떠도 정상이며, 무시하고 Cell 2부터 다시 실행하면 됩니다.**

`transformers`/`accelerate`는 좌측 패널의 선택적 로컬 LLM 브리핑 기능에 쓰입니다(끄면 규칙 기반 요약만 사용되며 항상 정상 동작합니다).

In [ ]:
!pip install -q --upgrade streamlit transformers accelerate "numpy<2"

import os
os.kill(os.getpid(), 9)  # 방금 설치한 패키지가 깨끗하게 적용되도록 런타임 자동 재시작


## Cell 2 — Streamlit 앱 코드 작성 (`app.py`)

In [ ]:
%%writefile app.py
# ============================================================================
# AOG(Aircraft on Ground) 자재 수급 어시스턴트 — Streamlit
# ----------------------------------------------------------------------------
# 좌측: AI(로컬 LLM 선택 사용) 상황 요약 & 추천 행동
# 우측: FAK -> Allocation -> Pooling -> Main Station 타사 -> 동일기종 타사 -> Hand-carry
#       6단계 프로세스를 사용자가 버튼으로 직접 확인하며 진행
# 사이드바: "AOG 대시보드" / "데이터 관리" 두 화면 전환 (각 데이터는 JSON 파일로 영속화)
# ============================================================================

import copy
import json
import os
import re
import urllib.parse
from datetime import datetime, timedelta

import pandas as pd
import streamlit as st

try:
    import transformers  # noqa: F401
    HAS_TRANSFORMERS = True
except ImportError:
    HAS_TRANSFORMERS = False

st.set_page_config(page_title="AOG 자재 수급 어시스턴트", page_icon="🚨", layout="wide")

DB_PATH = "aog_db.json"

# ----------------------------------------------------------------------------
# 1. 더미 데이터 (기본값) — 파일이 없을 때만 이 값으로 aog_db.json이 생성된다.
#    실제 운영 시에는 관리자가 "데이터 관리" 화면에서 언제든 갱신할 수 있다.
# ----------------------------------------------------------------------------

DEFAULT_DB = {
    "fak_stock": [
        {"aircraft_type": "A330-300", "part_number": "OXY-GEN-A330-15", "qty": 2, "location": "ICN FAK 창고 A-12"},
        {"aircraft_type": "B737-800", "part_number": "SMK-DET-737-04", "qty": 3, "location": "ICN FAK 창고 B-05"},
    ],
    "allocation_stock": [
        {"aircraft_type": "B777-300ER", "part_number": "BRK-B777-CARBON-01", "qty": 1, "location": "ICN 본사 Allocation 창고"},
        {"aircraft_type": "A330-300", "part_number": "FUEL-NOZ-A330-07", "qty": 4, "location": "ICN 본사 Allocation 창고"},
    ],
    "pooling_partners": [
        {"partner": "SIA Engineering (싱가포르)", "contact": "+65-6541-2000", "email": "poolingdesk@siae.com.sg"},
        {"partner": "Lufthansa Technik (독일)", "contact": "+49-40-5070-5553", "email": "pooling@lht.dlh.de"},
        {"partner": "ANA Base Maintenance (일본)", "contact": "+81-3-6735-1111", "email": "pooling@ana.co.jp"},
    ],
    "pooling_stock": [
        {"partner": "SIA Engineering (싱가포르)", "aircraft_type": "B737-800", "part_number": "HYD-PUMP-737-11", "qty": 1, "location": "SIN 창고"},
        {"partner": "Lufthansa Technik (독일)", "aircraft_type": "A330-300", "part_number": "IDG-A330-001", "qty": 1, "location": "FRA 창고"},
    ],
    "main_station_airlines": [
        {"airport": "ICN", "airline": "아시아나항공", "contact": "02-2669-8000", "email": "ops.icn@flyasiana.com"},
        {"airport": "GMP", "airline": "제주항공", "contact": "02-2015-1000", "email": "ops.gmp@jejuair.net"},
        {"airport": "NRT", "airline": "ANA", "contact": "+81-3-6735-1000", "email": "ops.nrt@ana.co.jp"},
        {"airport": "CDG", "airline": "Air France", "contact": "+33-1-4356-7890", "email": "ops.cdg@airfrance.fr"},
        {"airport": "JFK", "airline": "Delta Air Lines", "contact": "+1-800-221-1212", "email": "ops.jfk@delta.com"},
        {"airport": "SIN", "airline": "Singapore Airlines", "contact": "+65-6223-8888", "email": "ops.sin@singaporeair.com"},
    ],
    "other_airline_station_stock": [
        {"airport": "GMP", "part_number": "CAB-DOOR-ACT-321-03", "airline": "제주항공", "qty": 1,
         "location": "GMP", "contact": "02-2015-1000", "email": "ops.gmp@jejuair.net"},
    ],
    "fleet_operators": [
        {"aircraft_type": "A330-300", "airline": "아시아나항공", "contact": "02-2669-8000", "email": "fleet.a330@flyasiana.com"},
        {"aircraft_type": "A330-300", "airline": "Cathay Pacific", "contact": "+852-2747-1888", "email": "fleet.a330@cathaypacific.com"},
        {"aircraft_type": "A330-300", "airline": "China Eastern", "contact": "+86-21-95530", "email": "fleet.a330@ceair.com"},
        {"aircraft_type": "B777-300ER", "airline": "아시아나항공", "contact": "02-2669-8000", "email": "fleet.b777@flyasiana.com"},
        {"aircraft_type": "B777-300ER", "airline": "Emirates", "contact": "+971-600-555555", "email": "fleet.b777@emirates.com"},
        {"aircraft_type": "B777-300ER", "airline": "ANA", "contact": "+81-3-6735-1000", "email": "fleet.b777@ana.co.jp"},
        {"aircraft_type": "B737-800", "airline": "제주항공", "contact": "02-2015-1000", "email": "fleet.b737@jejuair.net"},
        {"aircraft_type": "B737-800", "airline": "티웨이항공", "contact": "1688-8686", "email": "fleet.b737@twayair.com"},
        {"aircraft_type": "B737-800", "airline": "Ryanair", "contact": "+353-1-945-1212", "email": "fleet.b737@ryanair.com"},
        {"aircraft_type": "A321neo", "airline": "에어부산", "contact": "1666-3060", "email": "fleet.a321n@airbusan.com"},
        {"aircraft_type": "A321neo", "airline": "IndiGo", "contact": "+91-124-6173838", "email": "fleet.a321n@goindigo.in"},
        {"aircraft_type": "A321neo", "airline": "Wizz Air", "contact": "+36-1-777-9300", "email": "fleet.a321n@wizzair.com"},
    ],
    "other_airline_fleet_stock": [
        {"aircraft_type": "B737-800", "part_number": "WHL-NLG-737-22", "airline": "티웨이항공", "qty": 1,
         "location": "ICN", "contact": "1688-8686", "email": "fleet.b737@twayair.com"},
    ],
    "allocation_dept_contacts": [
        {"airport": "ICN", "department": "자재관리팀 Allocation 파트", "contact": "02-XXXX-1000", "email": "allocation.icn@airline.example"},
        {"airport": "GMP", "department": "자재관리팀 Allocation 파트(김포)", "contact": "02-XXXX-1005", "email": "allocation.gmp@airline.example"},
    ],
    "customs_team": {"name": "통관팀", "contact": "02-XXXX-2000", "email": "customs@airline.example"},
    "flight_schedule": [
        {"destination_airport": "ICN", "flight_no": "KE901", "airline": "대한항공", "hours_from_now": 3},
        {"destination_airport": "ICN", "flight_no": "KE905", "airline": "대한항공", "hours_from_now": 7},
        {"destination_airport": "GMP", "flight_no": "KE1201", "airline": "대한항공", "hours_from_now": 2},
        {"destination_airport": "NRT", "flight_no": "KE705", "airline": "대한항공", "hours_from_now": 4},
        {"destination_airport": "CDG", "flight_no": "KE901", "airline": "대한항공", "hours_from_now": 9},
        {"destination_airport": "JFK", "flight_no": "KE081", "airline": "대한항공", "hours_from_now": 11},
        {"destination_airport": "SIN", "flight_no": "KE643", "airline": "대한항공", "hours_from_now": 6},
    ],
}

STEP_NAMES = {
    1: "1단계 · FAK(Fly Away Kit) 재고 확인",
    2: "2단계 · Allocation 재고 확인",
    3: "3단계 · Pooling 파트너사 재고 확인",
    4: "4단계 · Main Station 타 항공사 대여 요청",
    5: "5단계 · 동일 기종 타 항공사 대여 요청",
    6: "6단계 · 본사 직원 Hand-carry 이동",
}
TOTAL_STEPS = 6


# ----------------------------------------------------------------------------
# 2. 영속화 (JSON 파일 기반) — 관리자가 데이터 관리 화면에서 수정하면 즉시 저장되어
#    다음 실행/세션에도 유지된다.
# ----------------------------------------------------------------------------

def load_db():
    if os.path.exists(DB_PATH):
        try:
            with open(DB_PATH, "r", encoding="utf-8") as f:
                return json.load(f)
        except Exception:
            pass
    db = copy.deepcopy(DEFAULT_DB)
    save_db(db)
    return db


def save_db(db):
    with open(DB_PATH, "w", encoding="utf-8") as f:
        json.dump(db, f, ensure_ascii=False, indent=2)


if "db" not in st.session_state:
    st.session_state.db = load_db()
if "case" not in st.session_state:
    st.session_state.case = None
if "llm_briefing" not in st.session_state:
    st.session_state.llm_briefing = None


# ----------------------------------------------------------------------------
# 3. 조회 로직 — 사용자가 자유롭게 입력한 텍스트도 매칭되도록 대소문자/공백을 정규화한다.
# ----------------------------------------------------------------------------

def norm(s):
    return (s or "").strip().upper()


def tel_href(contact):
    return re.sub(r"[^0-9+]", "", contact or "")


def find_one(records, **filters):
    for r in records:
        if all(norm(r.get(k)) == norm(v) for k, v in filters.items()):
            return r
    return None


def evaluate_step(step, db, aircraft_type, part_number, airport):
    """단계별 재고/대여 확보 여부를 판단한다.
    반환값: (found: bool, message: str, detail: dict|None)
    detail에는 후속 액션 패널(전화/메일/편명 조회)에 필요한 원본 레코드를 담는다.
    """

    if step == 1:
        hit = find_one(db["fak_stock"], aircraft_type=aircraft_type, part_number=part_number)
        if hit:
            return True, (f"✅ **FAK 재고 확인됨**\n- 위치: {hit['location']} · 가용 수량: {hit['qty']}개\n\n"
                           f"당사 FAK 재고로 즉시 조치 가능합니다."), hit
        return False, "❌ FAK(Fly Away Kit)에는 해당 자재 재고가 없습니다.", None

    if step == 2:
        hit = find_one(db["allocation_stock"], aircraft_type=aircraft_type, part_number=part_number)
        if hit:
            return True, (f"✅ **Allocation 재고 확인됨**\n- 위치: {hit['location']} · 가용 수량: {hit['qty']}개\n\n"
                           f"당사 Allocation 재고로 조치 가능합니다."), hit
        return False, "❌ 당사 Allocation 재고에도 해당 자재가 없습니다.", None

    if step == 3:
        hit = find_one(db["pooling_stock"], aircraft_type=aircraft_type, part_number=part_number)
        if hit:
            partner_info = find_one(db["pooling_partners"], partner=hit["partner"]) or {}
            detail = {**hit, "contact": partner_info.get("contact", ""), "email": partner_info.get("email", "")}
            return True, (f"✅ **Pooling 파트너사 재고 확인됨 — {hit['partner']}**\n"
                           f"- 위치: {hit['location']} · 가용 수량: {hit['qty']}개\n"
                           f"- 연락처: {detail['contact']}\n\n"
                           f"Pooling 계약에 따라 대여 요청이 가능합니다."), detail
        checked = ", ".join(p["partner"] for p in db["pooling_partners"]) or "등록된 파트너사 없음"
        return False, f"❌ 조회한 Pooling 파트너사({checked}) 중 해당 자재 재고를 보유한 곳이 없습니다.", None

    if step == 4:
        queried = [r for r in db["main_station_airlines"] if norm(r["airport"]) == norm(airport)]
        hit = find_one(db["other_airline_station_stock"], airport=airport, part_number=part_number)
        detail = {"queried_airlines": queried, "hit": hit}
        if hit:
            return True, (f"✅ **Main Station 타 항공사 재고 확인됨 — {hit['airline']}**\n"
                           f"- 위치: {hit['location']} · 가용 수량: {hit['qty']}개\n- 연락처: {hit['contact']}\n\n"
                           f"{airport}을 Main Station으로 운영 중인 항공사로부터 대여 가능합니다."), detail
        op_names = ", ".join(o["airline"] for o in queried) if queried else "등록된 항공사 없음"
        return False, f"❌ {airport}을 Main Station으로 운영하는 항공사({op_names})에 문의했으나 재고가 없습니다.", detail

    if step == 5:
        queried = [r for r in db["fleet_operators"] if norm(r["aircraft_type"]) == norm(aircraft_type)]
        hit = find_one(db["other_airline_fleet_stock"], aircraft_type=aircraft_type, part_number=part_number)
        detail = {"queried_airlines": queried, "hit": hit}
        if hit:
            return True, (f"✅ **동일 기종 운영 타 항공사 재고 확인됨 — {hit['airline']}**\n"
                           f"- 위치: {hit['location']} · 가용 수량: {hit['qty']}개\n- 연락처: {hit['contact']}\n\n"
                           f"{aircraft_type} 기종을 운영 중인 타 항공사로부터 대여 가능합니다."), detail
        op_names = ", ".join(o["airline"] for o in queried) if queried else "등록된 항공사 없음"
        return False, f"❌ {aircraft_type} 운영 항공사({op_names})에 문의했으나 재고가 없습니다.", detail

    if step == 6:
        flights = sorted(
            [f for f in db["flight_schedule"] if norm(f["destination_airport"]) == norm(airport)],
            key=lambda f: f["hours_from_now"],
        )[:3]
        detail = {"flights": flights, "customs": db["customs_team"]}
        return True, ("🧳 **본사 직원 Hand-carry 이동 확정**\n"
                       "모든 사전 단계에서 재고를 찾지 못해 최종 수단으로 조치합니다. "
                       "통관팀과 가장 빠른 편명을 확인하세요."), detail

    raise ValueError(f"알 수 없는 단계: {step}")


# ----------------------------------------------------------------------------
# 4. AI 요약 — 규칙 기반(항상 동작, 기본값) + 로컬 LLM(선택, 실패해도 앱이 죽지 않음)
# ----------------------------------------------------------------------------

def build_rule_based_briefing(case):
    lines = [
        "### 📌 케이스 개요",
        f"- 기종: **{case['aircraft_type']}**  ·  자재 넘버: **{case['part_number']}**  ·  공항: **{case['airport']}**",
        f"- 현재 단계: **{case['step']}/{TOTAL_STEPS} — {STEP_NAMES[case['step']]}**",
        "",
        "### 🔍 지금까지 조회 결과",
    ]
    for rec in case["records"]:
        icon = "✅" if rec["found"] else "❌"
        lines.append(f"- {icon} {STEP_NAMES[rec['step']]}")

    lines.append("")
    lines.append("### 💡 추천 행동")

    last = case["records"][-1]
    if case["finished"]:
        if last["found"]:
            action = {
                1: "요청 정비사에게 FAK 재고 확보 사실을 재연락하세요.",
                2: "해당 지점 Allocation 부서에 연락해 자재 반출을 요청하세요.",
                3: "Pooling 파트너사에 연락해 대여 절차를 진행하세요.",
                4: "Main Station 항공사와 대여 조건을 확정하고, 조회 대상 항공사 전체에 상황 종료 메일을 보내는 것을 권장합니다.",
                5: "동일 기종 운영사와 대여 조건을 확정하고, 조회 대상 항공사 전체에 상황 종료 메일을 보내는 것을 권장합니다.",
                6: "통관팀에 연락하고, 가장 빠른 편명으로 Hand-carry를 확정하세요.",
            }[last["step"]]
            lines.append(f"- 🏁 **{STEP_NAMES[last['step']]}**에서 자재가 확보되었습니다. {action}")
        else:
            lines.append(f"- 🏁 담당자가 **{STEP_NAMES[last['step']]}** 단계에서 재고 미확보 상태로 프로세스를 수동 종료했습니다. 대체 조달 방안을 검토하세요.")
    else:
        if case["step"] >= TOTAL_STEPS:
            lines.append("- ▶ 이번 단계가 마지막 조달 경로입니다. Hand-carry 조치를 확정해주세요.")
        else:
            nxt = case["step"] + 1
            lines.append(f"- ▶ 현재 단계에서 재고를 찾지 못했다면 **{STEP_NAMES[nxt]}**로 진행하세요.")
        if case["step"] >= 4:
            lines.append("- ⚠️ 이미 4단계 이상 진행되어 지연 리스크가 높습니다. 병행해서 인접 항공사에 긴급 연락을 취하는 것을 권장합니다.")
    return "\n".join(lines)


@st.cache_resource(show_spinner=False)
def _load_local_llm():
    from transformers import pipeline
    return pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct")


def generate_llm_briefing(context_text):
    """로컬 LLM으로 상황을 다듬어 본다. 어떤 이유로든 실패하면 None을 반환해
    호출부가 규칙 기반 요약만으로도 정상 동작하도록 한다 (AI는 어디까지나 보조 기능)."""
    try:
        pipe = _load_local_llm()
        messages = [
            {"role": "system", "content": "당신은 항공사 AOG 자재 수급을 돕는 간결한 한국어 어시스턴트입니다."},
            {"role": "user", "content": f"다음 상황을 2~3문장으로 브리핑하고, 담당자가 지금 해야 할 행동을 한 문장으로 제안해줘:\n{context_text}"},
        ]
        out = pipe(messages, max_new_tokens=180, do_sample=False)
        content = out[0]["generated_text"][-1]["content"]
        return content.strip()
    except Exception:
        return None


# ----------------------------------------------------------------------------
# 5. 케이스(프로세스) 상태 관리
# ----------------------------------------------------------------------------

def start_case(aircraft_type, part_number, airport, mechanic_contact):
    case = {
        "aircraft_type": aircraft_type.strip(),
        "part_number": part_number.strip(),
        "airport": airport.strip(),
        "mechanic_contact": mechanic_contact.strip(),
        "step": 1,
        "finished": False,
        "records": [],
        "action_log": [],
        "started_at": datetime.now().strftime("%Y-%m-%d %H:%M"),
    }
    found, message, detail = evaluate_step(1, st.session_state.db, aircraft_type, part_number, airport)
    case["records"].append({"step": 1, "found": found, "message": message, "detail": detail})
    st.session_state.case = case
    st.session_state.llm_briefing = None


def proceed_step():
    case = st.session_state.case
    if case["finished"] or case["step"] >= TOTAL_STEPS:
        return
    nxt = case["step"] + 1
    found, message, detail = evaluate_step(nxt, st.session_state.db, case["aircraft_type"], case["part_number"], case["airport"])
    case["step"] = nxt
    case["records"].append({"step": nxt, "found": found, "message": message, "detail": detail})
    st.session_state.llm_briefing = None


def resolve_here():
    case = st.session_state.case
    if case["finished"]:
        return
    case["finished"] = True
    st.session_state.llm_briefing = None


# ----------------------------------------------------------------------------
# 6. 대시보드 화면 (메인) — 좌: AI 어시스턴트 / 우: 프로세스 진행 + 액션 패널
# ----------------------------------------------------------------------------

def render_case_intake():
    with st.form("case_form", clear_on_submit=False):
        c1, c2, c3 = st.columns(3)
        aircraft_type = c1.text_input("항공기 기종 (Aircraft Type)", placeholder="예: A330-300")
        part_number = c2.text_input("자재 넘버 (Part Number)", placeholder="예: OXY-GEN-A330-15")
        airport = c3.text_input("AOG 발생 공항 (Station)", placeholder="예: ICN")
        mechanic_contact = st.text_input("요청 정비사 연락처 (선택 — FAK 확보 시 재연락용)", placeholder="예: 010-1234-5678")
        submitted = st.form_submit_button("🚨 AOG 프로세스 시작", type="primary", use_container_width=True)

    if submitted:
        if not aircraft_type.strip() or not part_number.strip() or not airport.strip():
            st.warning("항공기 기종 / 자재 넘버 / 공항을 모두 입력해주세요.")
        else:
            start_case(aircraft_type, part_number, airport, mechanic_contact)
            st.rerun()


def render_ai_panel():
    st.markdown("## 🤖 AI 상황 요약 & 추천 행동")
    case = st.session_state.case
    if not case:
        st.info("좌측 하단이 아니라 위쪽 입력창에서 케이스를 시작하면 이곳에 AI 브리핑이 표시됩니다.")
        return

    with st.container(border=True):
        st.markdown(build_rule_based_briefing(case))

    if not HAS_TRANSFORMERS:
        st.caption("ℹ️ `transformers`가 설치되어 있지 않아 로컬 LLM 다듬기 기능은 비활성화되어 있습니다 (규칙 기반 요약은 정상 동작).")
        return

    with st.expander("🧠 로컬 LLM으로 브리핑 다듬기 (선택, 최초 1회 모델 로딩에 다소 시간이 걸릴 수 있음)"):
        if st.button("✨ AI 브리핑 생성/새로고침", key="gen_llm_briefing"):
            context = build_rule_based_briefing(case)
            with st.spinner("로컬 모델 추론 중..."):
                result = generate_llm_briefing(context)
            if result:
                st.session_state.llm_briefing = result
            else:
                st.session_state.llm_briefing = None
                st.warning("로컬 모델을 불러오지 못했습니다. 규칙 기반 요약을 계속 사용하세요 (네트워크 차단 시 정상적인 현상입니다).")
        if st.session_state.llm_briefing:
            st.markdown("**AI 코멘트**")
            st.markdown(st.session_state.llm_briefing)


def render_process_panel():
    st.markdown("## 📋 프로세스 진행")
    case = st.session_state.case
    if not case:
        st.info("케이스가 시작되지 않았습니다.")
        return

    last = case["records"][-1]
    with st.container(border=True):
        st.markdown(f"#### 진행 단계 {case['step']}/{TOTAL_STEPS} — {STEP_NAMES[case['step']]}")
        if last["found"]:
            st.success(last["message"])
        else:
            st.warning(last["message"])

        is_last_step = case["step"] >= TOTAL_STEPS
        bc1, bc2 = st.columns(2)
        if not case["finished"] and not is_last_step:
            if bc1.button("▶ 다음 단계 진행", use_container_width=True, key="proceed_btn"):
                proceed_step()
                st.rerun()
        if not case["finished"]:
            label = "✅ Hand-carry 조치 확정 (프로세스 종료)" if is_last_step else "✅ 현재 단계에서 조치 (프로세스 종료)"
            if bc2.button(label, use_container_width=True, key="resolve_btn", type="primary"):
                resolve_here()
                st.rerun()

    if case["finished"]:
        render_action_panel(case)

    with st.expander("📜 처리 이력 (Audit Log)", expanded=False):
        for rec in case["records"]:
            icon = "✅" if rec["found"] else "❌"
            st.markdown(f"**{icon} {STEP_NAMES[rec['step']]}**\n\n{rec['message']}")
            st.divider()
        for entry in case["action_log"]:
            st.markdown(f"🗂️ {entry}")


def render_action_panel(case):
    st.markdown("#### 🎯 다음 조치")
    last = case["records"][-1]
    step, found, detail = last["step"], last["found"], last["detail"]

    if not found:
        st.info("이 단계에서는 재고를 찾지 못한 채 담당자가 프로세스를 수동 종료했습니다. 대체 조달 방안을 별도로 검토해주세요.")
        return

    if step == 1:
        st.write(f"📍 위치: {detail['location']} · 수량: {detail['qty']}개")
        if case["mechanic_contact"]:
            href = tel_href(case["mechanic_contact"])
            st.markdown(f'<a href="tel:{href}">📞 요청 정비사({case["mechanic_contact"]})에게 전화하기</a>', unsafe_allow_html=True)
        else:
            st.caption("정비사 연락처가 입력되지 않았습니다. 위 정보로 직접 재연락해주세요.")
        if st.button("☎️ 재연락 완료로 기록", key="log_step1"):
            case["action_log"].append(f"[{datetime.now().strftime('%H:%M')}] 정비사에게 FAK 재고 확보 사실 재연락 완료")
            st.rerun()

    elif step == 2:
        dept = find_one(st.session_state.db["allocation_dept_contacts"], airport=case["airport"])
        st.write(f"📍 위치: {detail['location']} · 수량: {detail['qty']}개")
        if dept:
            st.write(f"담당 부서: **{dept['department']}**")
            colA, colB = st.columns(2)
            colA.markdown(f'<a href="tel:{tel_href(dept["contact"])}">📞 {dept["contact"]}</a>', unsafe_allow_html=True)
            colB.markdown(f'<a href="mailto:{dept["email"]}">✉️ {dept["email"]}</a>', unsafe_allow_html=True)
        else:
            st.warning(f"{case['airport']} 지점의 Allocation 부서 연락처가 등록되어 있지 않습니다. '데이터 관리'에서 추가해주세요.")
        if st.button("☎️ Allocation 부서 연락 완료로 기록", key="log_step2"):
            case["action_log"].append(f"[{datetime.now().strftime('%H:%M')}] {case['airport']} Allocation 부서에 연락 완료")
            st.rerun()

    elif step == 3:
        st.write(f"📍 위치: {detail['location']} · 수량: {detail['qty']}개")
        colA, colB = st.columns(2)
        colA.markdown(f'<a href="tel:{tel_href(detail["contact"])}">📞 {detail["contact"]}</a>', unsafe_allow_html=True)
        if detail.get("email"):
            colB.markdown(f'<a href="mailto:{detail["email"]}">✉️ {detail["email"]}</a>', unsafe_allow_html=True)
        if st.button("☎️ Pooling 파트너사 연락 완료로 기록", key="log_step3"):
            case["action_log"].append(f"[{datetime.now().strftime('%H:%M')}] {detail['partner']}에 대여 연락 완료")
            st.rerun()

    elif step in (4, 5):
        recipients = [a["email"] for a in detail["queried_airlines"] if a.get("email")]
        subject = f"[긴급] AOG 자재 지원 요청 - {case['aircraft_type']} / {case['part_number']} / {case['airport']}"
        body = (
            f"안녕하세요,\n\n{case['airport']} 공항에서 {case['aircraft_type']} 기종 AOG가 발생하여 "
            f"자재({case['part_number']}) 지원을 긴급 요청드립니다.\n재고 보유 여부를 확인해 회신 부탁드립니다.\n\n감사합니다."
        )
        st.write(f"조회 대상 항공사: {', '.join(a['airline'] for a in detail['queried_airlines']) or '없음'}")
        if recipients:
            mailto = "mailto:" + ",".join(recipients) + "?subject=" + urllib.parse.quote(subject) + "&body=" + urllib.parse.quote(body)
            st.markdown(f'<a href="{mailto}" target="_blank">✉️ 조회 대상 항공사 전체({len(recipients)}곳)에 긴급 메일 작성하기</a>', unsafe_allow_html=True)
        else:
            st.warning("등록된 이메일 주소가 없습니다. '데이터 관리'에서 항공사 이메일을 추가해주세요.")
        with st.expander("메일 내용 미리보기"):
            st.text(f"제목: {subject}\n\n{body}")
        if st.button("✉️ 긴급 메일 발송 완료로 기록", key=f"log_step{step}"):
            case["action_log"].append(f"[{datetime.now().strftime('%H:%M')}] {len(recipients)}개 항공사에 긴급 메일 발송 완료")
            st.rerun()

    elif step == 6:
        customs = detail["customs"]
        st.write(f"통관팀: **{customs['name']}** · {customs['contact']} · {customs['email']}")
        flights = detail["flights"]
        if flights:
            options = [
                f"{f['flight_no']} ({f['airline']}) — 약 {f['hours_from_now']}시간 후 도착 "
                f"[{(datetime.now() + timedelta(hours=f['hours_from_now'])).strftime('%m/%d %H:%M')} 예정]"
                for f in flights
            ]
            chosen = st.radio("Hand-carry로 활용할 편명을 선택하세요 (가장 빠른 편이 상단)", options, key="flight_choice")
            if st.button("✅ 이 편으로 Hand-carry 확정", key="log_step6"):
                case["action_log"].append(f"[{datetime.now().strftime('%H:%M')}] 통관팀 연계 완료, Hand-carry 편명 확정: {chosen}")
                st.rerun()
        else:
            st.warning(f"{case['airport']}행 등록된 편명이 없습니다. '데이터 관리'에서 편명 스케줄을 추가해주세요.")


def render_dashboard_page():
    st.title("🚨 AOG 자재 수급 어시스턴트")
    st.caption("항공기 기종 / 자재 넘버 / AOG 발생 공항을 직접 입력해 프로세스를 시작하세요.")

    render_case_intake()

    if st.session_state.case:
        st.divider()
        left, right = st.columns(2)
        with left:
            render_ai_panel()
        with right:
            render_process_panel()

        st.divider()
        if st.button("🔄 새 AOG 건 시작 (현재 케이스 초기화)"):
            st.session_state.case = None
            st.session_state.llm_briefing = None
            st.rerun()


# ----------------------------------------------------------------------------
# 7. 데이터 관리 화면 — 주기적으로 바뀌는 FAK/Allocation/Pooling/공항별 계약 등을
#    비개발자도 표 형태로 직접 수정할 수 있게 한다. 저장 시 JSON 파일에 즉시 반영된다.
# ----------------------------------------------------------------------------

def df_editor_save(key, columns, int_cols=()):
    db = st.session_state.db
    df = pd.DataFrame(db[key])
    if df.empty:
        df = pd.DataFrame(columns=columns)
    edited = st.data_editor(df, num_rows="dynamic", use_container_width=True, key=f"editor_{key}")
    if st.button("💾 저장", key=f"save_{key}"):
        cleaned = edited.fillna("")
        for c in int_cols:
            if c in cleaned.columns:
                cleaned[c] = pd.to_numeric(cleaned[c], errors="coerce").fillna(0).astype(int)
        db[key] = cleaned.to_dict("records")
        save_db(db)
        st.success("저장되었습니다.")


def render_admin_page():
    st.title("⚙️ 데이터 관리")
    st.caption("FAK/Allocation 재고, Pooling 계약, 공항·기종별 연락처 등은 주기적으로 바뀌므로 여기서 직접 갱신하세요. 저장 즉시 대시보드에 반영됩니다.")

    if os.path.exists(DB_PATH):
        mtime = datetime.fromtimestamp(os.path.getmtime(DB_PATH)).strftime("%Y-%m-%d %H:%M")
        st.caption(f"마지막 저장: {mtime}")

    tabs = st.tabs([
        "🏭 FAK 재고", "📦 Allocation 재고", "🤝 Pooling 파트너/재고",
        "🛫 공항별 Main Station/대여재고", "✈️ 기종별 운영사/대여재고",
        "📇 Allocation 부서·통관팀", "🛬 Hand-carry 편명 스케줄",
    ])

    with tabs[0]:
        df_editor_save("fak_stock", ["aircraft_type", "part_number", "qty", "location"], int_cols=["qty"])

    with tabs[1]:
        df_editor_save("allocation_stock", ["aircraft_type", "part_number", "qty", "location"], int_cols=["qty"])

    with tabs[2]:
        st.markdown("**파트너사 목록**")
        df_editor_save("pooling_partners", ["partner", "contact", "email"])
        st.markdown("**파트너사 보유 재고**")
        df_editor_save("pooling_stock", ["partner", "aircraft_type", "part_number", "qty", "location"], int_cols=["qty"])

    with tabs[3]:
        st.markdown("**공항별 Main Station 운영 항공사**")
        df_editor_save("main_station_airlines", ["airport", "airline", "contact", "email"])
        st.markdown("**공항별 대여 가능 재고 (4단계 매칭용)**")
        df_editor_save("other_airline_station_stock",
                        ["airport", "part_number", "airline", "qty", "location", "contact", "email"], int_cols=["qty"])

    with tabs[4]:
        st.markdown("**기종별 운영 항공사**")
        df_editor_save("fleet_operators", ["aircraft_type", "airline", "contact", "email"])
        st.markdown("**기종별 대여 가능 재고 (5단계 매칭용)**")
        df_editor_save("other_airline_fleet_stock",
                        ["aircraft_type", "part_number", "airline", "qty", "location", "contact", "email"], int_cols=["qty"])

    with tabs[5]:
        st.markdown("**공항별 Allocation 부서 연락처**")
        df_editor_save("allocation_dept_contacts", ["airport", "department", "contact", "email"])
        st.markdown("**통관팀 연락처**")
        db = st.session_state.db
        c1, c2, c3 = st.columns(3)
        name = c1.text_input("담당팀명", value=db["customs_team"]["name"], key="customs_name")
        contact = c2.text_input("연락처", value=db["customs_team"]["contact"], key="customs_contact")
        email = c3.text_input("이메일", value=db["customs_team"]["email"], key="customs_email")
        if st.button("💾 저장", key="save_customs"):
            db["customs_team"] = {"name": name, "contact": contact, "email": email}
            save_db(db)
            st.success("저장되었습니다.")

    with tabs[6]:
        df_editor_save("flight_schedule", ["destination_airport", "flight_no", "airline", "hours_from_now"], int_cols=["hours_from_now"])


# ----------------------------------------------------------------------------
# 8. 사이드바 내비게이션 — 메인(대시보드) / 데이터 관리 두 화면 전환
# ----------------------------------------------------------------------------

with st.sidebar:
    st.markdown("## ✈️ AOG 어시스턴트")
    page = st.radio("화면 선택", ["🚨 AOG 대시보드", "⚙️ 데이터 관리"], key="nav_page")
    st.divider()
    st.caption("Engine: Streamlit · 로컬 LLM(선택) · JSON 영속화")

if page == "🚨 AOG 대시보드":
    render_dashboard_page()
else:
    render_admin_page()


## Cell 3 — 앱 실행 (Colab 자체 포트포워딩만 사용, 외부 터널 불필요)

npm/localtunnel 같은 제3자 서비스를 전혀 쓰지 않고, Google Colab이 공식 제공하는 커널 포트포워딩(`google.colab.output`)만 사용합니다. 이미 접속 중인 `colab.research.google.com` 도메인을 그대로 경유하므로 사내망 보안 정책에 걸릴 일이 없습니다.

In [ ]:
import subprocess, time

LOG_PATH = "/content/streamlit_log.txt"

# 기존에 떠 있던 프로세스가 있으면 정리
subprocess.run(["pkill", "-f", "streamlit run app.py"], stderr=subprocess.DEVNULL)

process = subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
        "--server.enableWebsocketCompression", "false",
    ],
    stdout=open(LOG_PATH, "w"),
    stderr=subprocess.STDOUT,
)

booted = False
for _ in range(30):
    time.sleep(1)
    log_text = open(LOG_PATH).read()
    if "You can now view your Streamlit app" in log_text or "Local URL" in log_text:
        booted = True
        break

if not booted:
    print("Streamlit이 기동되지 않았습니다. 아래 로그를 확인하세요.\n")
    print(log_text)
else:
    from google.colab import output
    print("Streamlit 정상 기동. 아래에서 새 창으로 대시보드를 엽니다.")
    output.serve_kernel_port_as_window(8501)

    # 새 창 팝업이 막혀 있다면 아래 줄의 주석을 풀어 노트북 내부에 바로 임베드할 수 있습니다.
    # output.serve_kernel_port_as_iframe(8501, height=900)
